## Load trained model and eval on MoNLI


In [1]:
import os
import yaml
import torch

import json
import wandb

from collections import defaultdict
from typing import List, Tuple, Callable, Union, Dict


from robin_nlp.gpt_classification.train_gpt_text_classifier import GPTClassifier, parse_config
from robin_nlp.gpt_classification.dataset_config import get_dataset_config


from robin_nlp.gpt_classification.utils.utils_shortcut import *
from robin_nlp.actor_dataset_generator.generate_shortcut_dataset import *

from transformer_lens import HookedTransformer, ActivationCache


/home/feynman/Documents_Linux/shortcut_mechanisms/robin_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wandb.init(project="template" ,name="template", entity="template", mode="disabled")


In [3]:
# exp_name = "multi_nli_classifier"
exp_name = "snli_classifier"

result_folder = f'../results/{exp_name}/'


In [4]:
# def load_trained_model(config_path: str, model_path: str, dataset_path: str) -> Tuple[GPTClassifier, HookedTransformer, List[Dict[str, Union[str, bool]]]]:
def load_model(exp_name: str, result_path=None) -> Tuple[GPTClassifier, HookedTransformer, List[Dict[str, Union[str, bool]]]]:
    if result_path is None:
        result_path = "../results/" + exp_name + "/"

    # dataset_path = result_path + "processed_imdb_dataset.pkl"  # Update this path to where your dataset is saved
    model_path = result_path + "gpt2_classifier.pth"  # Update this path to where your model is saved
    config_path = result_path + "config.yml"

    args = parse_config(config_path)
    print(args.use_hooked_transform)
    logger = get_logger()

    dataset_config = get_dataset_config(args.dataset)

    classifier = GPTClassifier(args, logger, dataset_config)
    state_dict = torch.load(model_path)
    classifier.model.load_state_dict(state_dict)
    
    # Set the custom data (important for tokenizer and data loaders)
    classifier.model.eval()
    classifier.model.to("cuda")
    model: HookedTransformer = classifier.model

    return classifier, model


classifier, model = load_model(exp_name, result_path=result_folder)

True
Logging to: ./logs/process_and_train_20250403_133042.log
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda


### Load MoNLI dataset
- Currently the data is formated by the `Classifier` object, which is a bit annoying
- Therefore we need to process the data using `classifier.set_custom_data(train, validation, test)`
- Afterward, the data is loaded the right format as `classifier.dataloaders["split"]`. With where split can be replaced with : `train`, `test`, `val`, to obtain the right dataloader

In [5]:
# Run MoNLI evaluation
dataset_config = get_dataset_config("monli")
monli_train_data, monli_test_data, _, label_mapping = dataset_config.load_data()

In [6]:
# Will process each dataset separately
# MoNLI does not have a validation set, so we will use the test set for validation
classifier.set_custom_data(monli_train_data, monli_test_data, monli_test_data, label_mapping)

In [7]:
# Evaluate model on MoNLI datasets
monli_test_acc, results = classifier.evaluate(classifier.dataloaders["test"], True)
print("For MoNLI dataset - Test set:", monli_test_acc)

100%|██████████| 4/4 [00:00<00:00,  4.49it/s]

For MoNLI dataset - Test set: 0.0900000001


In [8]:
monli_train_acc, results = classifier.evaluate(classifier.dataloaders["train"], True)
print("For MoNLI dataset - Train set:", monli_train_acc)

100%|██████████| 32/32 [00:02<00:00, 11.72it/s]

For MoNLI dataset - Train set: 0.07584830341317365


## Finetune on MoNLI train set 1 epoch
- We already set the right datasets (although train and eval are the same dataset)
- Now we need to set the training settings
- The config from before is still loaded, so we will need to overwrite these from the loaded model

### Check the training args
- Many of these are likely not relevant for the new task (many are from IMDB shortcut dataset so should do nothing now)

In [9]:
printall = True
if printall:
    print("#### Results: ")
    for key, value in vars(classifier.args).items():
        print(f"{key}: {value}")
# vars()

#### Results: 
batch_size: 16
data_processing: {'num_actors': 1, 'sentence_window_size': 2, 'shortcut_only_full': True, 'start_name_idx': 0, 'test_imbalance': 0.1, 'train_imbalance': 0.001, 'train_purity': 1.0, 'val_imbalance': 0.05}
dataset: monli
epochs: 3
eval_batch_size: 32
eval_every: 1
exp_name: snli_classifier
gradient_accumulation_steps: 4
learning_rate: 2e-05
load_processed_data: False
manual_prepend_bos: True
max_tokens: 100
model_name: gpt2
num_workers: 2
paths: {'dataset_save_path': './results/snli_classifier/processed_dataset.pkl', 'model_save_path': './results/snli_classifier/gpt2_classifier.pth', 'output_dir': './results/', 'results_save_path': './results/snli_classifier/classification_results.json'}
recreate_data: True
sample_size: 0
save_model: False
save_processed_data: False
seed: 42
update_label_only: True
use_hooked_transform: True
use_wandbid_name: True
wandb_name: gpt_classifier


In [11]:
ft_epochts = 1
ft_lr = 1e-5

classifier.args.epochs = ft_epochts
classifier.args.learning_rate = ft_lr


In [ ]:
classifier.train(wandb)

/home/feynman/Documents_Linux/shortcut_mechanisms/robin_env/lib/python3.9/site-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Moving model to device:  cuda


100%|██████████| 4/4 [00:00<00:00,  7.18it/s]
2025-04-03 13:31:06,671 - robin_nlp.gpt_classification.utils.utils_shortcut - INFO - Epoch 1/1 - Train Loss: 0.4173 - Val Accuracy: 0.9700


In [13]:
# Evaluate model on MoNLI datasets
monli_test_acc, results = classifier.evaluate(classifier.dataloaders["test"], True)
print("For MoNLI dataset - Test set:", monli_test_acc)

100%|██████████| 4/4 [00:00<00:00,  7.14it/s]

For MoNLI dataset - Test set: 0.9700000001
